In [ ]:
# ==========================================
# 2. MOUNT DRIVE & LINK DATASET
# ==========================================
from google.colab import drive
import yaml

drive.mount('/content/drive')

DRIVE_DATASET_PATH = "/content/drive/MyDrive/clean_deduped_4classes"
LOCAL_DATASET_PATH = "/content/clean_deduped_4classes"

# Ensure local runtime dataset directory exists
if not os.path.exists(LOCAL_DATASET_PATH):
    print("Copying dataset from Google Drive to fast local runtime storage...")
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    print("✅ Dataset copied to local runtime!")
else:
    print("✅ Local dataset directory detected.")

# ==========================================
# 3. GENERATE FRESH DATA.YAML FILE
# ==========================================
yaml_path = f"{LOCAL_DATASET_PATH}/data.yaml"

# Construct explicit data configuration
data_cfg = {
    'path': LOCAL_DATASET_PATH,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {
        0: 'impacted tooth',
        1: 'Root Canal Treatment',
        2: 'Crown',
        3: 'Caries'
    }
}

with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"✅ Generated fresh '{yaml_path}' pointing to '{LOCAL_DATASET_PATH}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Local dataset directory detected.
✅ Generated fresh '/content/clean_deduped_4classes/data.yaml' pointing to '/content/clean_deduped_4classes'.


In [ ]:
import os
import shutil
import yaml
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

DRIVE_DATASET_PATH = "/content/drive/MyDrive/clean_deduped_4classes"
LOCAL_DATASET_PATH = "/content/clean_deduped_4classes"

# 2. Force re-copy dataset to ensure all splits (train, val/valid, test) exist
print("Syncing dataset from Google Drive...")
if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)

shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
print("✅ Dataset fully copied to local storage!")

# 3. Detect validation folder naming ('val' vs 'valid')
val_dir = "val/images" if os.path.exists(f"{LOCAL_DATASET_PATH}/val") else "valid/images"
test_dir = "test/images" if os.path.exists(f"{LOCAL_DATASET_PATH}/test") else "valid/images"

# 4. Generate verified data.yaml
data_cfg = {
    'path': LOCAL_DATASET_PATH,
    'train': 'train/images',
    'val': val_dir,
    'test': test_dir,
    'names': {
        0: 'impacted tooth',
        1: 'Root Canal Treatment',
        2: 'Crown',
        3: 'Caries'
    }
}

yaml_path = f"{LOCAL_DATASET_PATH}/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"✅ Created verified configuration at: '{yaml_path}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Syncing dataset from Google Drive...
✅ Dataset fully copied to local storage!
✅ Created verified configuration at: '/content/clean_deduped_4classes/data.yaml'


In [ ]:
# ==========================================
# 4. LAUNCH OPTIMIZED YOLO TRAINING
# ==========================================
from ultralytics import YOLO

# Load pre-trained YOLOv8x weights
model = YOLO('yolov8x.pt')

# Train with optimized resolution (800px) and max worker processes
results = model.train(
    data='/content/clean_deduped_4classes/data.yaml',
    epochs=100,
    imgsz=800,               # Dropped from 1024 to 800px for ~2.5x speed boost
    batch=16,                # 800px frees up VRAM to safely run batch=16
    workers=8,               # Maximize multi-core CPU threads for batch pre-fetching
    patience=20,             # Early stopping if validation mAP plateaus

    # --- Class Imbalance & Hyperparameters ---
    cls=1.5,                 # Scale classification loss to prioritize rare targets
    copy_paste=0.4,          # Synthetic minority class copy-paste augmentation
    mosaic=1.0,              # Combine 4 images for multi-scale context
    mixup=0.1,               # Blending for overlapping structures
    scale=0.5,               # Scale jittering for varied lesion sizes

    # --- Checkpoints & Output Directory in Drive ---
    project='/content/drive/MyDrive/dental_opg_training_runs',
    name='yolov8x_clean_800px',
    save=True,
    device=0
)

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/clean_deduped_4classes/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8x_clean_800px, nbs=64, nms=

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics